🤖 Agente Inteligente para Priorização de Leads Comerciais
---------------------------------------------------------------------
Desenvolver um agente inteligente capaz de analisar uma base de leads
e gerar automaticamente uma lista priorizada, identificando as empresas
com maior potencial de conversão para aquisição ou troca de sistemas ERP. A priorização é realizada por meio de inferência com modelos de IA,
considerando critérios estratégicos do contexto comercial. O projeto utiliza modelos de linguagem (LLMs) para avaliar cada lead
com base em múltiplos critérios, aplicando uma lógica de decisão
ponderada (pesos explícitos), simulando o raciocínio de um especialista
em vendas B2B.

---------------------------------------------------------------------

🔁 **Permite alternar entre diferentes provedores de IA**

*   Groq  → alta performance e baixa latência;
*   Gemini → robustez e recursos avançados do ecossistema Google.

---------------------------------------------------------------------

📊 **Saída Esperada**

score - empresa - justificativa - contato - telefone

In [1]:
# Instalar os pacotes
!pip install -q groq google-generativeai pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 13.5 MB/s eta 0:00:00


In [2]:
# Bibliotecas principais do projeto
import json
import re
import pandas as pd

# Acessos aos secrets do Google Colab
from google.colab import userdata

In [11]:
# ============================================================
# ESCOLHA MANUAL DO PROVEDOR
# Deixe apenas UM bloco ativo por vez
# ============================================================

# ------------------------------------------------------------
# OPÇÃO 1: GROQ
from groq import Groq
api_key = userdata.get("GROQ_API_KEY")
client = Groq(api_key=api_key)
PROVIDER = "groq"
model = "openai/gpt-oss-120b"

# ------------------------------------------------------------
# OPÇÃO 2: GEMINI
#from google.colab import userdata
#import google.generativeai as genai

#api_key = userdata.get("GEMINI_API_KEY")  # <- seu nome correto
#genai.configure(api_key=api_key)
#model = genai.GenerativeModel("gemini-1.5-flash")
#PROVIDER = "gemini"

In [12]:
# Carregar a base de dados ou dataset
df = pd.read_csv("leads.csv")
#df
df.head()
#df.sample(5)

,id_lead,empresa,segmento,Contato,Fone,cidade,uf,qtd_funcionarios,faturamento_estimado,erp_atual,interesse,origem_lead
0,1,AgroCampo,Varejo,Carlos Mendes,(62) 99124-3801,Anápolis,GO,215,médio,Planilhas,alto,outbound
1,2,MecSul,Distribuição,Rafael Oliveira,(62) 99317-4420,Goiânia,GO,36,baixo,Sem ERP,baixo,evento
2,3,MetalCenter,Varejo,Juliano Ferreira,(34) 99256-7813,Uberlândia,MG,100,alto,ERP simples,alto,inbound
3,4,LogExpress,Construção,André Souza,(34) 99183-2297,Uberlândia,MG,170,médio,Sem ERP,alto,site
4,5,AgroSul,Indústria,Marcos Pereira,(63) 99271-5542,Palmas,TO,241,baixo,ERP simples,baixo,inbound


In [13]:
# Montar o prompt
def montar_prompt(dados_para_analise):
    prompt = f"""
Você é um especialista comercial em venda de ERP para empresas.

Analise a base de leads abaixo e atribua um score de prioridade de 0 a 100
para cada empresa, do maior potencial de compra ou troca de ERP para o menor.

UTILIZE EXPLICITAMENTE OS SEGUINTES PESOS:
- Interesse declarado: 30%
- ERP atual: 25%
- Porte da empresa: 20%
- Faturamento estimado: 15%
- Origem do lead: 10%

REGRAS IMPORTANTES:
- Avalie cada critério individualmente
- Combine os critérios de forma ponderada
- Retorne SOMENTE JSON válido
- Não use markdown
- Não escreva explicações antes ou depois
- A saída deve começar com {{ e terminar com }}

Formato esperado:
{{
  "leads": [
    {{
      "id_lead": 1,
      "empresa": "Nome da empresa",
      "score": 95,
      "justificativa": "Justificativa curta e objetiva"
    }}
  ]
}}

Base de leads:
{json.dumps(dados_para_analise, ensure_ascii=False)}
"""
    return prompt

In [14]:
def gerar_resposta(prompt, provider, model):
    """
    Envia o prompt para o provedor escolhido e retorna o texto da resposta.
    """

    if provider == "groq":
        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "user", "content": prompt}
            ],
            temperature=0.2
        )
        return response.choices[0].message.content

    elif provider == "gemini":
        response = client.models.generate_content(
            model=model,
            contents=prompt
        )
        return response.text

    else:
        raise ValueError("Provider inválido. Use 'groq' ou 'gemini'.")

In [15]:
def extrair_json(texto):
    """
    Tenta converter a resposta do modelo em JSON válido.
    Também trata casos em que o modelo retorna blocos markdown ou texto extra.
    """

    if not texto or not texto.strip():
        raise ValueError("O modelo retornou uma resposta vazia.")

    texto = texto.strip()

    # Remove blocos markdown, se existirem
    if texto.startswith("```json"):
        texto = texto.replace("```json", "", 1).strip()
    if texto.startswith("```"):
        texto = texto.replace("```", "", 1).strip()
    if texto.endswith("```"):
        texto = texto[:-3].strip()

    # Tenta converter diretamente
    try:
        return json.loads(texto)
    except json.JSONDecodeError:
        # Tenta extrair apenas o trecho JSON da resposta
        match = re.search(r'\{.*\}', texto, re.DOTALL)
        if match:
            return json.loads(match.group(0))
        raise ValueError(f"Não foi possível extrair um JSON válido.\n\nResposta recebida:\n{texto}")

In [16]:
# Preparar os dados para análise
dados_para_analise = df.fillna("").to_dict(orient="records")

# Montar o prompt
prompt = montar_prompt(dados_para_analise)

# Gerar resposta do modelo
conteudo = gerar_resposta(prompt, PROVIDER, model)

# Exibir resposta bruta para conferência
print("Resposta bruta do modelo:\n")
print(conteudo)

Resposta bruta do modelo:

{
  "leads": [
    {
      "id_lead": 1,
      "empresa": "AgroCampo",
      "score": 89,
      "justificativa": "Alto interesse, ERP básico (planilhas) e grande porte elevam a prioridade."
    },
    {
      "id_lead": 2,
      "empresa": "MecSul",
      "score": 39,
      "justificativa": "Baixo interesse apesar de não ter ERP; pequeno porte reduz o potencial."
    },
    {
      "id_lead": 3,
      "empresa": "MetalCenter",
      "score": 82,
      "justificativa": "Alto interesse, faturamento alto e inbound, porém usa ERP simples."
    },
    {
      "id_lead": 4,
      "empresa": "LogExpress",
      "score": 90,
      "justificativa": "Alto interesse, sem ERP, bom porte e origem site geram alta pontuação."
    },
    {
      "id_lead": 5,
      "empresa": "AgroSul",
      "score": 50,
      "justificativa": "Baixo interesse, porém grande porte e inbound compensam parcialmente."
    },
    {
      "id_lead": 6,
      "empresa": "Distribuidora Central",
  

In [17]:
# Converter resposta em JSON
resultado = extrair_json(conteudo)

# Transformar a lista de leads em DataFrame
df_scores = pd.DataFrame(resultado["leads"])

# Visualizar resultado bruto da IA
df_scores.head()

,id_lead,empresa,score,justificativa
0,1,AgroCampo,89,"Alto interesse, ERP básico (planilhas) e grand..."
1,2,MecSul,39,Baixo interesse apesar de não ter ERP; pequeno...
2,3,MetalCenter,82,"Alto interesse, faturamento alto e inbound, po..."
3,4,LogExpress,90,"Alto interesse, sem ERP, bom porte e origem si..."
4,5,AgroSul,50,"Baixo interesse, porém grande porte e inbound ..."


In [18]:
# Unir os scores gerados pela IA com dados originais da planilha
df_saida = df_scores.merge(
    df[["id_lead", "empresa", "Contato", "Fone"]],
    on=["id_lead", "empresa"],
    how="left"
)

# Ordenar do maior score para o menor
df_saida = df_saida.sort_values(by="score", ascending=False).reset_index(drop=True)

# Visualizar resultado final
df_saida

,id_lead,empresa,score,justificativa,Contato,Fone
0,4,LogExpress,90,"Alto interesse, sem ERP, bom porte e origem si...",André Souza,(34) 99183-2297
1,1,AgroCampo,89,"Alto interesse, ERP básico (planilhas) e grand...",Carlos Mendes,(62) 99124-3801
2,3,MetalCenter,82,"Alto interesse, faturamento alto e inbound, po...",Juliano Ferreira,(34) 99256-7813
3,6,Distribuidora Central,73,"Alto interesse e ERP antigo, mas porte pequeno...",Eduardo Martins,(64) 99142-6678
4,7,Distribuidora CentroOeste,67,"Interesse médio, ERP simples e grande porte, o...",Felipe Carvalho,(61) 99388-1204
5,9,Comercial Brasil Norte,53,"Baixo interesse, mas faturamento alto e porte ...",Thiago Alves,(64) 99167-8895
6,5,AgroSul,50,"Baixo interesse, porém grande porte e inbound ...",Marcos Pereira,(63) 99271-5542
7,2,MecSul,39,Baixo interesse apesar de não ter ERP; pequeno...,Rafael Oliveira,(62) 99317-4420
8,10,AgroForte,32,Baixo interesse e ERP consolidado; pequeno por...,Pedro Assunção,(63) 98745-9087
9,8,LogMaster,29,"Baixo interesse e pequeno porte, apesar de ERP...",Bruno Ribeiro,(62) 99205-7731


Se você chegou até aqui, parabéns! 🎆 🔥

Fim!